In [ ]:
!pip -q install pandas requests matplotlib gradio

In [ ]:
import requests
import sqlite3
import pandas as pd
import re
import string
import time
from collections import Counter
from datetime import datetime
import matplotlib.pyplot as plt
import unittest
import gradio as gr

In [ ]:
BASE_URL = "https://api.jikan.moe/v4"
DB_NAME = "anime_pipeline.db"

# You can change these
SEARCH_QUERY = "naruto"     # Example keyword search
MAX_PAGES = 3               # Keep moderate for demo
REQUEST_DELAY = 1.2

In [1]:
def clean_text(text):
    """
    Clean text by:
    - converting to lowercase
    - removing punctuation
    - removing HTML tags
    - removing extra spaces
    """
    if text is None or pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def safe_int(value, default=0):
    try:
        if value is None:
            return default
        return int(value)
    except:
        return default


def safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        return float(value)
    except:
        return default


def classify_score_band(score):
    """
    Classifying score into bands.
    """

    if score < 6:
        return "low"
    elif score > 6 and score < 8:
        return "medium"
    else:
        return "high"


def classify_popularity_band(popularity):
    """
    Lower popularity number means more popular in MAL/Jikan ranking terms.
    """
    if popularity is None or popularity == 0:
        return "unknown"
    popularity = safe_int(popularity, 0)
    if popularity <= 1000:
        return "very popular"
    elif popularity <= 2000:
        return "popular"
    elif popularity <= 5000:
        return "moderate"
    else:
        return "not popular"


def classify_episode_group(episodes):
    if episodes is None or episodes == 0:
        return "unknown"
    episodes = safe_int(episodes, 0)
    if episodes <= 15:
        return "short"
    elif episodes <= 26:
        return "medium"
    else:
        return "long"


def extract_names_from_list(items):
    if not items:
        return []

    return [item["name"] for item in items if isinstance(item, dict) and "name" in item]


def count_items_in_csv(text):

    if not text:
        return 0
    return len([x.strip() for x in text.split(",") if x.strip()])


In [2]:
def fetch_anime_search(query, max_pages=1, delay=2.0):
    """
    Fetch anime data from Jikan search endpoint.
    Endpoint example: /anime?q=naruto&page=1
    """
    all_anime = []

    for page in range(1, max_pages + 1):
        url = f"{BASE_URL}/anime"
        params = {
            "q": query,
            "page": page
        }

        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        page_data = data.get("data", [])
        if not page_data:
            break

        all_anime.extend(page_data)

        pagination = data.get("pagination", {})
        has_next_page = pagination.get("has_next_page", False)

        if not has_next_page:
            break

        time.sleep(delay)

    return all_anime


def fetch_top_anime(max_pages=1, delay=1.2):
    """
    Fetch top anime from Jikan top endpoint.
    """
    all_anime = []

    for page in range(1, max_pages + 1):
        url = f"{BASE_URL}/top/anime"
        params = {
            "page": page
        }

        response = requests.get(url, params=params, timeout=60)
        data = response.json()

        page_data = data.get("data", [])
        if not page_data:
            break

        all_anime.extend(page_data)

        pagination = data.get("pagination", {})
        has_next_page = pagination.get("has_next_page", False)

        if not has_next_page:
            break

        time.sleep(delay)

    return all_anime
